# So CLIP và RegionCLIP trên cùng box
Bật Internet. CPU chạy được; không cần Dataset hay secret. Gửi file regionclip_vs_clip_same_boxes_valid2.json.


In [ ]:
CASES=[{'coco_id': 239894, 'rois': [{'roi': 1, 'label': 'teddy bear', 'confidence': 0.926, 'bbox': [0.5461971830985916, 0.43859375, 0.793169014084507, 0.704875]}, {'roi': 6, 'label': 'handbag', 'confidence': 0.5338, 'bbox': [0.16483568075117372, 0.733359375, 0.4577230046948357, 0.9990937499999999]}], 'image_sha256': '79cfff40fc14c76c7bbe9a409b91ef1b03ec7759cb8e6ea94b277e49f810828b', 'feature_shape': [2, 1024], 'resized_xyxy': [[436.9577464788733, 527.1896875, 634.5352112676056, 847.25975], [131.86854460093898, 881.49796875, 366.17840375586854, 1200.9106875]], 'regionclip': {'coco_id': 239894, 'prompt': 'a teddy bear', 'matched': 0.08968288194702365, 'distractor': 0.05626174883934446, 'margin': 0.033421133107679184}}, {'coco_id': 487438, 'rois': [{'roi': 1, 'label': 'bed', 'confidence': 0.8526, 'bbox': [0.160296875, 0.48352083333333334, 0.7308125000000001, 0.9886666666666667]}, {'roi': 7, 'label': 'chair', 'confidence': 0.3943, 'bbox': [0.910171875, 0.6130208333333333, 0.999578125, 0.9525208333333333]}], 'image_sha256': '1921b1a7256c483720117c5aca5bc1ce67dfd73fff8132fbec639167edfd77b7', 'feature_shape': [2, 1024], 'resized_xyxy': [[171.03676562500002, 386.81666666666666, 779.7769375, 790.9333333333334], [971.153390625, 490.4166666666667, 1066.549859375, 762.0166666666667]], 'regionclip': {'coco_id': 487438, 'prompt': 'a bed', 'matched': 0.14632310692927192, 'distractor': 0.13712127191048532, 'margin': 0.009201835018786603}}]
import hashlib, io, json, math, urllib.request
from pathlib import Path
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL='openai/clip-vit-base-patch16'
REVISION='57c216476eefef5ab752ec549e440a49ae4ae5f3'
device='cuda' if torch.cuda.is_available() else 'cpu'
processor=CLIPProcessor.from_pretrained(MODEL,revision=REVISION)
model=CLIPModel.from_pretrained(MODEL,revision=REVISION,use_safetensors=False).to(device).eval()
results=[]
with torch.inference_mode():
    for case in CASES:
        cid=case['coco_id']
        url=f'https://s3.amazonaws.com/images.cocodataset.org/val2014/COCO_val2014_{cid:012d}.jpg'
        with urllib.request.urlopen(url,timeout=30) as response:
            raw=response.read()
        if hashlib.sha256(raw).hexdigest()!=case['image_sha256']:
            raise ValueError(f'Image hash mismatch for {cid}')
        image=Image.open(io.BytesIO(raw)).convert('RGB')
        width,height=image.size
        crops=[]
        pixel_boxes=[]
        for roi in case['rois']:
            x,y,x2,y2=roi['bbox']
            box=(math.floor(x*width),math.floor(y*height),
                 math.ceil(x2*width),math.ceil(y2*height))
            if not (0<=box[0]<box[2]<=width and 0<=box[1]<box[3]<=height):
                raise ValueError(f'Invalid crop for {cid}')
            crops.append(image.crop(box))
            pixel_boxes.append(box)
        region=case['regionclip']
        batch=processor(text=[region['prompt']],images=crops,return_tensors='pt',padding=True).to(device)
        output=model(**batch)
        cosine=(output.logits_per_image[:,0].float()/model.logit_scale.exp().float()).cpu().tolist()
        if len(cosine)!=2 or any(not math.isfinite(x) or abs(x)>1.00001 for x in cosine):
            raise ValueError('Invalid CLIP cosine')
        results.append(dict(coco_id=cid,prompt=region['prompt'],pixel_boxes=pixel_boxes,
            clip_crop=dict(matched=cosine[0],distractor=cosine[1],margin=cosine[0]-cosine[1]),
            regionclip=dict(matched=region['matched'],distractor=region['distractor'],margin=region['margin'])))
rank_counts={arm:sum(row[arm]['margin']>0 for row in results) for arm in ('clip_crop','regionclip')}
out=Path('/kaggle/working/regionclip_vs_clip_same_boxes_valid2.json')
out.write_text(json.dumps(dict(split='val',model=MODEL,revision=REVISION,device=device,images=len(results),
                               correct_box_ranked_first=rank_counts,records=results),indent=2,allow_nan=False),encoding='utf8')
print(out.read_text(encoding='utf8'))
print('SEND:',out)
